In [ ]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D2 — VINCI Consolidated Income Statement 2024
# ============================================================

!pip install pymupdf -q

import hashlib
import json
import platform
import re
import sys
from pathlib import Path

import fitz
import pandas as pd

from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.0 MB/s eta 0:00:00


In [ ]:
# ------------------------------------------------------------
# 1. Source upload and configuration
# ------------------------------------------------------------

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError("Upload exactly one PDF file for D2.")

SOURCE_PATH = pdf_files[0]

DOCUMENT_ID = "D2"
DOCUMENT_NAME = "VINCI Consolidated Income Statement 2024"

OUTPUT_DIR = Path(f"outputs_{DOCUMENT_ID}_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RECORD_COUNT = 22
EXPECTED_YEARS = ["2024", "2023"]

REFERENCE_COLUMNS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023",
    "Source Location"
]

EXTRACTION_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

print(f"Loaded source file: {SOURCE_PATH.name}")
print(f"Stage 1 output directory: {OUTPUT_DIR}")

Saving D2 - 2024-vinci-Income-Statement.pdf to D2 - 2024-vinci-Income-Statement.pdf
Loaded source file: D2 - 2024-vinci-Income-Statement.pdf
Stage 1 output directory: outputs_D2_stage1


In [ ]:
# ------------------------------------------------------------
# 2. Source identity and file hash
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

print(f"Source file: {SOURCE_PATH.name}")
print(f"SHA-256: {SOURCE_SHA256}")

Source file: D2 - 2024-vinci-Income-Statement.pdf
SHA-256: 6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667


In [ ]:
# ------------------------------------------------------------
# 3. PDF loading and text extraction
# ------------------------------------------------------------

doc = fitz.open(SOURCE_PATH)

page_texts = [
    page.get_text("text")
    for page in doc
]

full_text = "\n".join(page_texts)

document_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_filename": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "file_format": SOURCE_PATH.suffix.replace(".", "").upper(),
    "number_of_pages": len(doc),
    "text_extractable": bool(full_text.strip()),
    "total_text_characters": len(full_text),
    "total_text_words": len(full_text.split()),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "pymupdf_version": fitz.version[0]
}

print(
    json.dumps(
        document_metadata,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D2",
  "document_name": "VINCI Consolidated Income Statement 2024",
  "source_filename": "D2 - 2024-vinci-Income-Statement.pdf",
  "source_file_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
  "file_format": "PDF",
  "number_of_pages": 1,
  "text_extractable": true,
  "total_text_characters": 1324,
  "total_text_words": 170,
  "python_version": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "pandas_version": "2.2.3",
  "pymupdf_version": "1.28.2"
}


In [ ]:
# ------------------------------------------------------------
# 4. Document-level characterisation
# ------------------------------------------------------------

numeric_pattern = r"\(?-?\d+(?:,\d{3})*(?:\.\d+)?\)?"
numeric_tokens = re.findall(numeric_pattern, full_text)

word_count = len(full_text.split())

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)

percentage_tokens = re.findall(
    r"-?\d+(?:\.\d+)?%",
    full_text
)

non_empty_lines = [
    line.strip()
    for line in full_text.splitlines()
    if line.strip()
]

detected_years = sorted(
    set(re.findall(r"\b20\d{2}\b", full_text)),
    reverse=True
)

document_characterisation = {
    "document_id": DOCUMENT_ID,
    "page_count": len(doc),
    "text_extractable": bool(full_text.strip()),
    "native_text_available": bool(full_text.strip()),
    "ocr_dependency": "Low" if full_text.strip() else "High",
    "non_empty_line_count": len(non_empty_lines),
    "numeric_token_count": len(numeric_tokens),
    "numeric_token_to_word_ratio":
        round(numeric_token_to_word_ratio, 3),
    "percentage_token_count": len(percentage_tokens),
    "contains_financial_table": True,
    "contains_multi_year_values": True,
    "detected_years": detected_years,
    "contains_subtotals_and_emphasised_rows": True,
    "explicit_hierarchy_present": True,
    "negative_values_presented_with_parentheses": True,
    "multiple_measurement_units_present": True,
    "measurement_units": [
        "EUR millions",
        "EUR"
    ]
}

print(
    json.dumps(
        document_characterisation,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D2",
  "page_count": 1,
  "text_extractable": true,
  "native_text_available": true,
  "ocr_dependency": "Low",
  "non_empty_line_count": 72,
  "numeric_token_count": 47,
  "numeric_token_to_word_ratio": 0.276,
  "percentage_token_count": 0,
  "contains_financial_table": true,
  "contains_multi_year_values": true,
  "detected_years": [
    "2024",
    "2023"
  ],
  "contains_subtotals_and_emphasised_rows": true,
  "explicit_hierarchy_present": true,
  "negative_values_presented_with_parentheses": true,
  "multiple_measurement_units_present": true,
  "measurement_units": [
    "EUR millions",
    "EUR"
  ]
}


In [ ]:
# ------------------------------------------------------------
# 5. Extraction task definition
# ------------------------------------------------------------

EXTRACTION_TASK = """
Extract every line-item observation from the consolidated income
statement contained in the document.

For each observation, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source,
  including footnote markers and unit text contained in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in € millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.
- Return the two yearly values as numerical values.
- Do not calculate, infer, reconstruct, aggregate or correct any value.
- Use null only when a requested value is not available.
- Return one record for every visible line item.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
"""

print(EXTRACTION_TASK)


Extract every line-item observation from the consolidated income
statement contained in the document.

For each observation, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source,
  including footnote markers and unit text contained in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in € millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.
- Return the two yearly values as numerical values.
- Do not calculate, infer, reconstruct, aggregate or correct any value.
- Use null only when a requested value is not available.
- Return one record for every visible line item.
- Return only valid JSON.
- Do not include exp

In [ ]:
# ------------------------------------------------------------
# 6. Extraction and reference schemas
# ------------------------------------------------------------

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": "consolidated_income_statement_line_item",
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "fields": {
        "Line Item": {
            "type": ["string", "null"],
            "description": "Exact visible income-statement row label"
        },
        "Unit": {
            "type": ["string", "null"],
            "allowed_values": [
                "EUR millions",
                "EUR"
            ]
        },
        "Value 2024": {
            "type": ["number", "null"],
            "description": "Reported numerical value for 2024"
        },
        "Value 2023": {
            "type": ["number", "null"],
            "description": "Reported numerical value for 2023"
        }
    },
    "expected_output_structure": {
        "document_id": DOCUMENT_ID,
        "records": [
            {
                "Line Item": "string or null",
                "Unit": "EUR millions, EUR or null",
                "Value 2024": "number or null",
                "Value 2023": "number or null"
            }
        ]
    }
}

REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": "consolidated_income_statement_line_item",
    "extraction_fields": EXTRACTION_FIELDS,
    "reference_metadata_fields": [
        "Source Location"
    ],
    "matching_key": [
        "Line Item"
    ],
    "expected_record_count": EXPECTED_RECORD_COUNT
}

print("Extraction schema:")
print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

print("\nReference schema:")
print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

Extraction schema:
{
  "document_id": "D2",
  "record_level": "consolidated_income_statement_line_item",
  "expected_record_count": 22,
  "fields": {
    "Line Item": {
      "type": [
        "string",
        "null"
      ],
      "description": "Exact visible income-statement row label"
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ],
      "allowed_values": [
        "EUR millions",
        "EUR"
      ]
    },
    "Value 2024": {
      "type": [
        "number",
        "null"
      ],
      "description": "Reported numerical value for 2024"
    },
    "Value 2023": {
      "type": [
        "number",
        "null"
      ],
      "description": "Reported numerical value for 2023"
    }
  },
  "expected_output_structure": {
    "document_id": "D2",
    "records": [
      {
        "Line Item": "string or null",
        "Unit": "EUR millions, EUR or null",
        "Value 2024": "number or null",
        "Value 2023": "number or null"
      }
    ]
  }

In [ ]:
# ------------------------------------------------------------
# 7. Reference dataset construction
# ------------------------------------------------------------

reference_values = [
    {
        "Line Item": "Revenue (*)",
        "Unit": "EUR millions",
        "Value 2024": 71623,
        "Value 2023": 68838,
        "Source Location": "Page 1"
    },
    {
        "Line Item": (
            "Concession subsidiaries’ revenue derived from works "
            "carried out by non-Group companies"
        ),
        "Unit": "EUR millions",
        "Value 2024": 837,
        "Value 2023": 780,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Total revenue",
        "Unit": "EUR millions",
        "Value 2024": 72459,
        "Value 2023": 69619,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Revenue from ancillary activities",
        "Unit": "EUR millions",
        "Value 2024": 308,
        "Value 2023": 267,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Operating expenses",
        "Unit": "EUR millions",
        "Value 2024": -63770,
        "Value 2023": -61529,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Operating income from ordinary activities",
        "Unit": "EUR millions",
        "Value 2024": 8997,
        "Value 2023": 8357,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Share-based payments (IFRS 2)",
        "Unit": "EUR millions",
        "Value 2024": -462,
        "Value 2023": -360,
        "Source Location": "Page 1"
    },
    {
        "Line Item": (
            "Profit/(loss) of companies accounted for under "
            "the equity method"
        ),
        "Unit": "EUR millions",
        "Value 2024": 219,
        "Value 2023": 111,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Other recurring operating items",
        "Unit": "EUR millions",
        "Value 2024": 97,
        "Value 2023": 68,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Recurring operating income",
        "Unit": "EUR millions",
        "Value 2024": 8850,
        "Value 2023": 8175,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Non-recurring operating items",
        "Unit": "EUR millions",
        "Value 2024": -68,
        "Value 2023": -105,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Operating income",
        "Unit": "EUR millions",
        "Value 2024": 8783,
        "Value 2023": 8071,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Cost of gross financial debt",
        "Unit": "EUR millions",
        "Value 2024": -1785,
        "Value 2023": -1363,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Financial income from cash investments",
        "Unit": "EUR millions",
        "Value 2024": 595,
        "Value 2023": 469,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Cost of net financial debt",
        "Unit": "EUR millions",
        "Value 2024": -1191,
        "Value 2023": -894,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Other financial income and expense",
        "Unit": "EUR millions",
        "Value 2024": -217,
        "Value 2023": -157,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Income tax expense",
        "Unit": "EUR millions",
        "Value 2024": -2102,
        "Value 2023": -1917,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Net income",
        "Unit": "EUR millions",
        "Value 2024": 5274,
        "Value 2023": 5102,
        "Source Location": "Page 1"
    },
    {
        "Line Item": (
            "Net income attributable to non-controlling interests"
        ),
        "Unit": "EUR millions",
        "Value 2024": 410,
        "Value 2023": 400,
        "Source Location": "Page 1"
    },
    {
        "Line Item": (
            "Net income attributable to owners of the parent"
        ),
        "Unit": "EUR millions",
        "Value 2024": 4863,
        "Value 2023": 4702,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Basic earnings per share (in €)",
        "Unit": "EUR",
        "Value 2024": 8.53,
        "Value 2023": 8.28,
        "Source Location": "Page 1"
    },
    {
        "Line Item": "Diluted earnings per share (in €)",
        "Unit": "EUR",
        "Value 2024": 8.43,
        "Value 2023": 8.18,
        "Source Location": "Page 1"
    }
]

reference_values_df = pd.DataFrame(
    reference_values,
    columns=REFERENCE_COLUMNS
)

reference_values_df

,Line Item,Unit,Value 2024,Value 2023,Source Location
0,Revenue (*),EUR millions,71623.00,68838.00,Page 1
1,Concession subsidiaries’ revenue derived from ...,EUR millions,837.00,780.00,Page 1
2,Total revenue,EUR millions,72459.00,69619.00,Page 1
3,Revenue from ancillary activities,EUR millions,308.00,267.00,Page 1
4,Operating expenses,EUR millions,-63770.00,-61529.00,Page 1
5,Operating income from ordinary activities,EUR millions,8997.00,8357.00,Page 1
6,Share-based payments (IFRS 2),EUR millions,-462.00,-360.00,Page 1
7,Profit/(loss) of companies accounted for under...,EUR millions,219.00,111.00,Page 1
8,Other recurring operating items,EUR millions,97.00,68.00,Page 1
9,Recurring operating income,EUR millions,8850.00,8175.00,Page 1


In [ ]:
# ------------------------------------------------------------
# 8. Reference structure validation
# ------------------------------------------------------------

missing_columns = [
    column
    for column in REFERENCE_COLUMNS
    if column not in reference_values_df.columns
]

additional_columns = [
    column
    for column in reference_values_df.columns
    if column not in REFERENCE_COLUMNS
]

reference_schema_valid = (
    not missing_columns
    and not additional_columns
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_RECORD_COUNT
)

duplicate_line_items = int(
    reference_values_df.duplicated(
        subset=["Line Item"],
        keep=False
    ).sum()
)

missing_values_by_column = (
    reference_values_df
    .isna()
    .sum()
    .astype(int)
    .to_dict()
)

allowed_units = {
    "EUR millions",
    "EUR"
}

observed_units = set(
    reference_values_df["Unit"]
    .dropna()
    .unique()
)

unexpected_units = sorted(
    observed_units - allowed_units
)

numeric_columns_valid = all(
    pd.api.types.is_numeric_dtype(
        reference_values_df[column]
    )
    for column in [
        "Value 2024",
        "Value 2023"
    ]
)

source_locations_valid = (
    reference_values_df["Source Location"]
    .eq("Page 1")
    .all()
)

print("Reference schema valid:", reference_schema_valid)
print("Record count valid:", record_count_valid)
print("Duplicate line-item observations:", duplicate_line_items)
print("Missing values:", missing_values_by_column)
print("Unexpected units:", unexpected_units)
print("Numeric columns valid:", numeric_columns_valid)
print("Source locations valid:", source_locations_valid)

Reference schema valid: True
Record count valid: True
Duplicate line-item observations: 0
Missing values: {'Line Item': 0, 'Unit': 0, 'Value 2024': 0, 'Value 2023': 0, 'Source Location': 0}
Unexpected units: []
Numeric columns valid: True
Source locations valid: True


In [ ]:
# ------------------------------------------------------------
# 9. Unit assignment validation
# ------------------------------------------------------------

eps_mask = (
    reference_values_df["Line Item"]
    .str.contains(
        "earnings per share",
        case=False,
        na=False
    )
)

eps_records = reference_values_df[eps_mask]
main_statement_records = reference_values_df[~eps_mask]

eps_unit_valid = bool(
    len(eps_records) == 2
    and eps_records["Unit"].eq("EUR").all()
)

main_unit_valid = bool(
    len(main_statement_records) == 20
    and main_statement_records[
        "Unit"
    ].eq("EUR millions").all()
)

print("EPS observations:", len(eps_records))
print("EPS units valid:", eps_unit_valid)

print(
    "Main statement observations:",
    len(main_statement_records)
)
print(
    "Main statement units valid:",
    main_unit_valid
)

EPS observations: 2
EPS units valid: True
Main statement observations: 20
Main statement units valid: True


In [ ]:
# ------------------------------------------------------------
# 10. Negative-value validation
# ------------------------------------------------------------
expected_negative_items = {
    "Operating expenses",
    "Share-based payments (IFRS 2)",
    "Non-recurring operating items",
    "Cost of gross financial debt",
    "Cost of net financial debt",
    "Other financial income and expense",
    "Income tax expense"
}

negative_records = reference_values_df[
    reference_values_df["Line Item"]
    .isin(expected_negative_items)
].copy()

negative_values_valid = bool(
    len(negative_records) == len(expected_negative_items)
    and (
        negative_records[
            ["Value 2024", "Value 2023"]
        ] < 0
    ).all().all()
)

unexpected_negative_records = reference_values_df[
    (
        reference_values_df[
            ["Value 2024", "Value 2023"]
        ] < 0
    ).any(axis=1)
    & ~reference_values_df[
        "Line Item"
    ].isin(expected_negative_items)
]

print(
    "Expected negative observations found:",
    len(negative_records)
)

print(
    "Negative-value conversion valid:",
    negative_values_valid
)

print(
    "Unexpected negative observations:",
    len(unexpected_negative_records)
)

Expected negative observations found: 7
Negative-value conversion valid: True
Unexpected negative observations: 0


In [ ]:
# ------------------------------------------------------------
# 11. Expected line-item coverage
# ------------------------------------------------------------

expected_line_items = [
    "Revenue (*)",
    (
        "Concession subsidiaries’ revenue derived from works "
        "carried out by non-Group companies"
    ),
    "Total revenue",
    "Revenue from ancillary activities",
    "Operating expenses",
    "Operating income from ordinary activities",
    "Share-based payments (IFRS 2)",
    (
        "Profit/(loss) of companies accounted for under "
        "the equity method"
    ),
    "Other recurring operating items",
    "Recurring operating income",
    "Non-recurring operating items",
    "Operating income",
    "Cost of gross financial debt",
    "Financial income from cash investments",
    "Cost of net financial debt",
    "Other financial income and expense",
    "Income tax expense",
    "Net income",
    "Net income attributable to non-controlling interests",
    "Net income attributable to owners of the parent",
    "Basic earnings per share (in €)",
    "Diluted earnings per share (in €)"
]

expected_line_item_set = set(expected_line_items)

observed_line_item_set = set(
    reference_values_df["Line Item"]
)

missing_line_items = sorted(
    expected_line_item_set
    - observed_line_item_set
)

unexpected_line_items = sorted(
    observed_line_item_set
    - expected_line_item_set
)

line_item_coverage_valid = (
    not missing_line_items
    and not unexpected_line_items
)

print(
    "Line-item coverage valid:",
    line_item_coverage_valid
)

print(
    "Missing line items:",
    missing_line_items
)

print(
    "Unexpected line items:",
    unexpected_line_items
)

Line-item coverage valid: True
Missing line items: []
Unexpected line items: []


In [ ]:
# ------------------------------------------------------------
# 12. Reference summary
# ------------------------------------------------------------

reference_summary = {
    "document_id": DOCUMENT_ID,
    "number_of_reference_records": int(
        len(reference_values_df)
    ),
    "source_file_sha256": SOURCE_SHA256,
    "source_locations_valid": bool(
        source_locations_valid
    ),
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "record_count_valid": bool(record_count_valid),
    "years": EXPECTED_YEARS,
    "units": sorted(
        reference_values_df[
            "Unit"
        ].unique().tolist()
    ),
    "number_of_eur_million_records": int(
        reference_values_df[
            "Unit"
        ].eq("EUR millions").sum()
    ),
    "number_of_eur_records": int(
        reference_values_df[
            "Unit"
        ].eq("EUR").sum()
    ),
    "missing_values_by_column": (
        missing_values_by_column
    ),
    "duplicate_line_items": duplicate_line_items,
    "reference_schema_valid": bool(
        reference_schema_valid
    ),
    "numeric_columns_valid": bool(
        numeric_columns_valid
    ),
    "unit_assignment_valid": bool(
        eps_unit_valid
        and main_unit_valid
    ),
    "negative_value_conversion_valid": bool(
        negative_values_valid
    ),
    "line_item_coverage_valid": bool(
        line_item_coverage_valid
    ),
    "reference_values_branch_independent": True,
    "constructed_before_llm_extraction": True,
    "missing_line_items": missing_line_items,
    "unexpected_line_items": unexpected_line_items
}

print(
    json.dumps(
        reference_summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D2",
  "number_of_reference_records": 22,
  "source_file_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
  "source_locations_valid": true,
  "expected_record_count": 22,
  "record_count_valid": true,
  "years": [
    "2024",
    "2023"
  ],
  "units": [
    "EUR",
    "EUR millions"
  ],
  "number_of_eur_million_records": 20,
  "number_of_eur_records": 2,
  "missing_values_by_column": {
    "Line Item": 0,
    "Unit": 0,
    "Value 2024": 0,
    "Value 2023": 0,
    "Source Location": 0
  },
  "duplicate_line_items": 0,
  "reference_schema_valid": true,
  "numeric_columns_valid": true,
  "unit_assignment_valid": true,
  "negative_value_conversion_valid": true,
  "line_item_coverage_valid": true,
  "reference_values_branch_independent": true,
  "constructed_before_llm_extraction": true,
  "missing_line_items": [],
  "unexpected_line_items": []
}


In [ ]:
# ------------------------------------------------------------
# 13. Indicator-level quality assessment
# ------------------------------------------------------------

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Low",
        "Evidence Source":
            "PDF text extraction + manual inspection",
        "Justification":
            "The document contains a single income-statement table "
            "with a clear top-to-bottom sequence of line items."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Low",
        "Evidence Source":
            "Manual table inspection",
        "Justification":
            "The table has a regular structure consisting of one "
            "line-item column and two clearly aligned year columns, "
            "with no nested or complex table structures."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Low",
        "Evidence Source":
            "PDF text extraction + manual inspection",
        "Justification":
            "The document has a simple and clearly identifiable "
            "hierarchy consisting of the financial-statement heading, "
            "income-statement title, unit label, and year headers."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "The single-page PDF is visually sharp and all relevant "
            "text and numerical values are clearly readable."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "No relevant scanning noise, degradation, or visual "
            "artefacts affect document readability."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated PDF text extraction",
        "Justification":
            "Relevant document content is embedded as machine-readable "
            "text and can be extracted directly without OCR."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual content inspection",
        "Justification":
            "Financial and accounting terminology is used consistently "
            "throughout the income statement."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Low",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The visible line-item labels and the 2024 and 2023 value "
            "columns correspond directly to the fields required by the "
            "extraction task, while units are explicitly recoverable "
            "from the document."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "High",
        "Evidence Source":
            "Automated text profiling + manual inspection",
        "Justification":
            "The document is dominated by quantitative financial "
            "observations, with two reported numerical values for "
            "each of the 22 income-statement line items."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All 22 expected line items contain the information "
            "required by the extraction task for both reporting years."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Low",
        "Evidence Source":
            "Reference-value and source inspection",
        "Justification":
            "The two-year reporting structure, financial notation, "
            "and measurement conventions are applied consistently "
            "throughout the income statement."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "Low",
        "Evidence Source":
            "Manual document inspection",
        "Justification":
            "The document contains a single compact financial table "
            "with one dominant representation format."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "Medium",
        "Evidence Source":
            "Manual representation inspection",
        "Justification":
            "Most observations use EUR millions, while the two "
            "earnings-per-share rows use EUR; negative financial "
            "amounts are represented using parentheses and one "
            "line-item label contains a footnote marker."
    }
]

indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

indicator_assessment_df

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Low,PDF text extraction + manual inspection,The document contains a single income-statemen...
1,Structural Readiness,Table Structure Integrity,Low,Manual table inspection,The table has a regular structure consisting o...
2,Structural Readiness,Section/Header Hierarchy,Low,PDF text extraction + manual inspection,The document has a simple and clearly identifi...
3,Visual/OCR Readiness,Sharpness,Low,Manual visual inspection,The single-page PDF is visually sharp and all ...
4,Visual/OCR Readiness,Noise / Degradation,Low,Manual visual inspection,"No relevant scanning noise, degradation, or vi..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated PDF text extraction,Relevant document content is embedded as machi...
6,Semantic Quality,Terminology Consistency,Low,Manual content inspection,Financial and accounting terminology is used c...
7,Semantic Quality,Schema Alignment,Low,Reference-schema comparison,The visible line-item labels and the 2024 and ...
8,Semantic Quality,Numerical Density,High,Automated text profiling + manual inspection,The document is dominated by quantitative fina...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All 22 expected line items contain the informa...


In [ ]:
# ------------------------------------------------------------
# 14. Quality-assessment validation
# ------------------------------------------------------------

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: "
        f"{invalid_scores}"
    )


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: "
        f"{missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: "
        f"{unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [ ]:
# ------------------------------------------------------------
# 15. Dimension-level assessment
# ------------------------------------------------------------

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

dimension_assessment_df

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.00,2,Low
1,Representation and Normalisation Complexity,1.50,2,Medium
2,Semantic Quality,1.67,3,Medium
3,Structural Readiness,1.00,3,Low
4,Visual/OCR Readiness,1.00,3,Low


In [ ]:
# ------------------------------------------------------------
# 16. Reference integrity check
# ------------------------------------------------------------

all_reference_checks_passed = all([
    source_locations_valid,
    reference_schema_valid,
    record_count_valid,
    duplicate_line_items == 0,
    all(
        value == 0
        for value
        in missing_values_by_column.values()
    ),
    not unexpected_units,
    numeric_columns_valid,
    eps_unit_valid,
    main_unit_valid,
    negative_values_valid,
    len(unexpected_negative_records) == 0,
    line_item_coverage_valid
])

reference_integrity = {
    "document_id": DOCUMENT_ID,
    "all_checks_passed": bool(
        all_reference_checks_passed
    ),
    "checks": {
        "reference_schema_valid": bool(
            reference_schema_valid
        ),
        "record_count_valid": bool(
            record_count_valid
        ),
        "no_duplicate_line_items": bool(
            duplicate_line_items == 0
        ),
        "no_missing_reference_values": bool(
            all(
                value == 0
                for value
                in missing_values_by_column.values()
            )
        ),
        "source_locations_valid": bool(
            source_locations_valid
        ),
        "only_expected_units": bool(
            not unexpected_units
        ),
        "numeric_columns_valid": bool(
            numeric_columns_valid
        ),
        "unit_assignment_valid": bool(
            eps_unit_valid
            and main_unit_valid
        ),
        "negative_value_conversion_valid": bool(
            negative_values_valid
        ),
        "no_unexpected_negative_records": bool(
            len(unexpected_negative_records) == 0
        ),
        "complete_line_item_coverage": bool(
            line_item_coverage_valid
        )
    }
}

print(
    json.dumps(
        reference_integrity,
        indent=2,
        ensure_ascii=False
    )
)

if not all_reference_checks_passed:
    raise ValueError(
        "One or more D2 reference-value checks failed."
    )

{
  "document_id": "D2",
  "all_checks_passed": true,
  "checks": {
    "reference_schema_valid": true,
    "record_count_valid": true,
    "no_duplicate_line_items": true,
    "no_missing_reference_values": true,
    "source_locations_valid": true,
    "only_expected_units": true,
    "numeric_columns_valid": true,
    "unit_assignment_valid": true,
    "negative_value_conversion_valid": true,
    "no_unexpected_negative_records": true,
    "complete_line_item_coverage": true
  }
}


In [ ]:
# ------------------------------------------------------------
# 17. Structured quality-assessment evidence
# ------------------------------------------------------------

quality_evidence = {
    "document_id": DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },
        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",
        "classification_rule": {
            "Low": "mean < 1.5",
            "Medium": "1.5 <= mean < 2.5",
            "High": "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

print(
    json.dumps(
        quality_evidence,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D2",
  "assessment_basis": "Observed document evidence was mapped to the predefined Low, Medium, and High operational criteria defined in Table 3.3 of the methodology.",
  "evidence_method": "Evidence was obtained through automated profiling where measurable characteristics could be derived programmatically and through documented manual inspection where qualitative assessment was required.",
  "indicators": [
    {
      "Dimension": "Structural Readiness",
      "Indicator": "Reading Order Quality",
      "Score": "Low",
      "Evidence Source": "PDF text extraction + manual inspection",
      "Justification": "The document contains a single income-statement table with a clear top-to-bottom sequence of line items."
    },
    {
      "Dimension": "Structural Readiness",
      "Indicator": "Table Structure Integrity",
      "Score": "Low",
      "Evidence Source": "Manual table inspection",
      "Justification": "The table has a regular structure consisting of one 

In [ ]:
# ------------------------------------------------------------
# 18. Stage 1 artefact export
# ------------------------------------------------------------

REFERENCE_VALUES_CSV_PATH = (
    OUTPUT_DIR / "D2_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR / "D2_reference_values.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR / "D2_reference_schema.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D2_extraction_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR / "D2_reference_summary.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR / "D2_reference_integrity.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D2_document_characterisation.json"
)

DOCUMENT_METADATA_PATH = (
    OUTPUT_DIR / "D2_document_metadata.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D2_extraction_task.txt"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR / "D2_quality_evidence.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D2_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D2_dimension_assessment.csv"
)


reference_values_df.to_csv(
    REFERENCE_VALUES_CSV_PATH,
    index=False
)

reference_values_df.to_json(
    REFERENCE_VALUES_JSON_PATH,
    orient="records",
    indent=2,
    force_ascii=False
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False
)

with open(
    EXTRACTION_SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        EXTRACTION_SCHEMA,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    REFERENCE_SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        REFERENCE_SCHEMA,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    REFERENCE_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        reference_summary,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    REFERENCE_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        reference_integrity,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    QUALITY_EVIDENCE_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        quality_evidence,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    DOCUMENT_CHARACTERISATION_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        document_characterisation,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    DOCUMENT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        document_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

with open(
    EXTRACTION_TASK_PATH,
    "w",
    encoding="utf-8"
) as file:
    file.write(
        EXTRACTION_TASK.strip()
    )

print("D2 outputs exported successfully.")

D2 outputs exported successfully.


In [ ]:
# ------------------------------------------------------------
# 19. Final notebook summary
# ------------------------------------------------------------

summary = {
    "document_id": DOCUMENT_ID,
    "document_metadata": document_metadata,
    "document_characterisation": (
        document_characterisation
    ),
    "reference_summary": reference_summary,
    "reference_integrity": (
        reference_integrity
    ),
    "indicator_assessment": (
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        )
    ),
    "dimension_assessment": (
        dimension_assessment_df.to_dict(
            orient="records"
        )
    ),
    "outputs_created": [
        path.name
        for path in [
            REFERENCE_VALUES_CSV_PATH,
            REFERENCE_VALUES_JSON_PATH,
            INDICATOR_ASSESSMENT_PATH,
            DIMENSION_ASSESSMENT_PATH,
            EXTRACTION_SCHEMA_PATH,
            REFERENCE_SCHEMA_PATH,
            REFERENCE_SUMMARY_PATH,
            REFERENCE_INTEGRITY_PATH,
            QUALITY_EVIDENCE_PATH,
            DOCUMENT_CHARACTERISATION_PATH,
            DOCUMENT_METADATA_PATH,
            EXTRACTION_TASK_PATH
        ]
    ]
}

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D2",
  "document_metadata": {
    "document_id": "D2",
    "document_name": "VINCI Consolidated Income Statement 2024",
    "source_filename": "D2 - 2024-vinci-Income-Statement.pdf",
    "source_file_sha256": "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667",
    "file_format": "PDF",
    "number_of_pages": 1,
    "text_extractable": true,
    "total_text_characters": 1324,
    "total_text_words": 170,
    "python_version": "3.13.15",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "pandas_version": "2.2.3",
    "pymupdf_version": "1.28.2"
  },
  "document_characterisation": {
    "document_id": "D2",
    "page_count": 1,
    "text_extractable": true,
    "native_text_available": true,
    "ocr_dependency": "Low",
    "non_empty_line_count": 72,
    "numeric_token_count": 47,
    "numeric_token_to_word_ratio": 0.276,
    "percentage_token_count": 0,
    "contains_financial_table": true,
    "contains_multi_year_values": true,
    "detect

In [ ]:
# ------------------------------------------------------------
# 20. Download Stage 1 outputs
# ------------------------------------------------------------

download_paths = [
    REFERENCE_VALUES_CSV_PATH,
    REFERENCE_VALUES_JSON_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_INTEGRITY_PATH,
    QUALITY_EVIDENCE_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    DOCUMENT_METADATA_PATH,
    EXTRACTION_TASK_PATH
]

for output_path in download_paths:
    files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>